In [3]:
import os
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time

# ============ 配置 ============
client = OpenAI(
    base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
    api_key='ida_TbtBzRyj8HeV7kmxkeRsgImHnYelRIUMQ3vW9VIf'
)

MODEL = 'qwen-2.5-72b-instruct'

# ============ Few-Shot Prompt ============
FEW_SHOT_PROMPT_TEMPLATE = """You are a professional search system optimization expert. Your task is to decide whether to apply Pseudo-Relevance Feedback (PRF).

### Instructions
1. **Analyze the Query Intent:** What exactly is the user looking for? (Fact, Definition, Broad Topic, etc.)
2. **Audit Retrieved Documents:** Are the top results relevant? Do they cover the *entire* intent? Are there false matches?
3. **Check for "Poison Pills":** If a highly-ranked document contains factually incorrect or dangerously misleading information (e.g., wrong war, wrong person), output **NO**. PRF is blind and will extract these errors, contaminating the query.
4. **Check for Saturation:** If the top results already perfectly answer the query (High Precision), PRF is risky/unnecessary. Output **NO** to avoid "hallucinating" gaps that don't exist.
5. **Check for Drift:** If the results are mixed or irrelevant, PRF will add noise. Output **NO**.
6. **Identify Opportunity:** Only output **YES** if the results are relevant but *incomplete* or *vague*, and PRF could add useful vocabulary to improve recall.

Here are examples of how to analyze:

### Example 1 (Scenario: Clear Benefit)
Query: "definition of a sigmet"
Retrieved Documents: [Top-4 docs consistently define it as an aviation weather advisory. Doc-5 is a medical false match "sigmoid" but ranked low.]
Analysis:
* **User Intent:** Specific definition of the aviation term "SIGMET".
* **Result Quality:** High. Documents consistently identify it as a weather advisory.
* **Saturation Check:** The definition is present, but short.
* **PRF Forecast:** **Positive**. PRF would likely extract related technical terms like "turbulence", "icing", "convective", and "pilot safety". These are consistently present in the good docs and would help reinforce the aviation context against the medical false match.
Decision: YES

### Example 2 (Scenario: Disambiguation)
Query: "who is robert gray"
Retrieved Documents: [Docs about Explorer Robert Gray vs Politician Robert Gray vs Game Show Host...]
Analysis:
* **User Intent:** Biographical information. Ambiguous name, but likely seeking the historical explorer based on entity dominance in top results.
* **Result Quality:** Mixed/Ambiguous. Top results favor the explorer, but modern politicians and celebrities also appear.
* **PRF Forecast:** **Positive**. PRF can extract terms unique to the explorer (e.g., "Columbia River", "Circumnavigate", "Captain", "Oregon"). This helps "anchor" the query to the historical figure and filter out the noise from the other "Robert Grays".
Decision: YES

### Example 3 (Scenario: Catastrophic Drift)
Query: "what is theraderm used for"
Retrieved Documents: [1-2 about skincare, 3-5 about thermocouples, therabands, thermometers...]
Analysis:
* **User Intent:** Specific product information (skincare).
* **Result Quality:** **Catastrophic Failure**. The retrieval system matched prefixes ("thera-", "thermo-") instead of the semantic concept. Only 2 docs are relevant; 3 are completely off-topic (Physics/Fitness).
* **PRF Forecast:** **DANGEROUS**. PRF would blindly extract words from the top ranked docs, mixing "lanolin" (skincare) with "voltage" (physics) and "resistance" (fitness). This would destroy the query semantics.
Decision: NO

### Example 4 (Scenario: Poison Pill / Contamination)
Query: "reasons for us entry into ww1"
Retrieved Documents:
[1] The sinking of the Lusitania and the Zimmerman Telegram pushed the US into WWI.
[2] The attack on **Pearl Harbor** by Japan led to immediate US entry into the war.
[3] Unrestricted submarine warfare was a key factor for WWI.
Analysis:
* **User Intent:** Historical reasons for **WWI**.
* **Result Quality:** **Dangerous Mix**. While Docs 1 and 3 are correct, Doc 2 discusses **WWII** (Pearl Harbor) with high confidence.
* **PRF Forecast:** **NEGATIVE**. PRF algorithms are blind; they will extract terms from *all* top documents. This means terms like "Pearl Harbor", "Japan", and "1941" will be added to the query. This is a "poison pill" that will contaminate the search, confusing WWI with WWII.
Decision: NO

### Example 5 (Scenario: Saturation / Perfect Result)
Query: "what is famvir prescribed for"
Retrieved Documents: [All 5 docs clearly list: Genital herpes, cold sores, shingles, chickenpox...]
Analysis:
* **User Intent:** Factoid (Medical Indications).
* **Result Quality:** **Saturated**. All top documents are highly relevant, authoritative, and answer the question completely.
* **PRF Forecast:** **Unnecessary**. The user's need is fully satisfied. The model might be tempted to add terms like "dosage", "side effects", or "mechanism of action" to make it "more complete", but the user did not ask for these. Adding them dilutes the focus on the simple list of conditions.
Decision: NO

---
Now analyze the following query using the structured format above:

Query: "{query}"
Retrieved Documents:
{documents}

Analysis:
* **User Intent:**
* **Result Quality:**
* **PRF Forecast:**
Decision:
"""

# ============ 加载数据 ============
print("加载数据...")
df_preference = pd.read_csv('preference.csv')
df_queries = pd.read_csv('result_with_text.csv')
df_colbert = pd.read_csv('df_colbert_deduped.csv')

if 'query_text' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query_text'].first().to_dict()
elif 'query' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query'].first().to_dict()

user_study_qids = df_preference['qid'].tolist()[:15]
print(f"测试 User Study 的前 10 个查询")

# ============ 生成并打印完整输出 ============
results = []

for i, qid in enumerate(user_study_qids, 1):
    print("\n" + "="*80)
    print(f"查询 {i}/10 - QID: {qid}")
    print("="*80)
    
    query = qid_to_query.get(qid, f"[Query not found for QID {qid}]")
    print(f"Query: {query}")
    
    # 获取真实标签
    ground_truth_row = df_preference[df_preference['qid'] == qid]
    if len(ground_truth_row) > 0:
        ground_truth_preference = ground_truth_row['preference'].iloc[0]
        ground_truth_ratio = ground_truth_row['b_preference_ratio'].iloc[0]
        print(f"Ground Truth: {ground_truth_preference} (ratio: {ground_truth_ratio:.3f})")
    else:
        ground_truth_preference = "Unknown"
        ground_truth_ratio = None
        print(f"Ground Truth: Unknown")
    
    # 获取 Top-5
    top5 = df_colbert[df_colbert['qid'] == qid].nlargest(5, 'score')
    
    if len(top5) == 0:
        print(f"⚠️ 没有检索结果")
        continue
    
    print(f"\nTop-5 文档:")
    for idx, (_, row) in enumerate(top5.iterrows(), 1):
        print(f"  [{idx}] Score: {row['score']:.2f}")
        print(f"      {str(row['passage_text'])[:150]}...")
    
    # 格式化文档
    docs_text = "\n".join([
        f"[{i+1}] [Score: {row['score']:.2f}] {str(row['passage_text'])[:250]}..."
        for i, (_, row) in enumerate(top5.iterrows())
    ])
    
    # 填充 prompt
    prompt = FEW_SHOT_PROMPT_TEMPLATE.format(
        query=query,
        documents=docs_text
    )
    
    print(f"\n{'='*80}")
    print("🤖 Teacher Model 输出:")
    print("="*80)
    
    # 调用 API
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=1000
        )
        
        output = response.choices[0].message.content
        
        # 直接打印完整输出
        print(output)
        
        # 保存结果
        results.append({
            'qid': qid,
            'query': query,
            'ground_truth_preference': ground_truth_preference,
            'ground_truth_ratio': ground_truth_ratio,
            'teacher_output': output
        })
        
    except Exception as e:
        print(f"✗ API 调用失败: {e}")
        results.append({
            'qid': qid,
            'query': query,
            'ground_truth_preference': ground_truth_preference,
            'ground_truth_ratio': ground_truth_ratio,
            'error': str(e)
        })
    
    print("\n" + "="*80)
    
    # 延迟
    time.sleep(1)

# ============ 保存完整输出 ============
df_results = pd.DataFrame(results)
df_results.to_csv('teacher_outputs_10_queries_full.csv', index=False, encoding='utf-8')

print("\n" + "="*80)
print("✓ 测试完成！")
print("="*80)
print(f"结果已保存到: teacher_outputs_10_queries_full.csv")
print(f"成功生成: {len([r for r in results if 'teacher_output' in r])}/{len(results)}")
print("\n现在你可以:")
print("1. 查看上面的输出，评估 Teacher 的推理质量")
print("2. 检查 Analysis 是否详细、合理")
print("3. 看看 Decision 是否与 Ground Truth 一致")
print("4. 如果质量好，再扩展到全部 6980 个查询")

加载数据...
测试 User Study 的前 10 个查询

查询 1/10 - QID: 104861
Query: cost of interior concrete flooring
Ground Truth: PRF-Benefit (ratio: 0.713)

Top-5 文档:
  [1] Score: 25.10
      Even though each project is unique, interior building finishes for commercial buildings typically fall in an. average cost per square foot range. $5-$...
  [2] Score: 24.07
      We pour a lot of 3000 psi. concrete for interior concrete floors and 4000 psi concrete for exterior concrete. The concrete cost per yard of concrete i...
  [3] Score: 23.25
      How Much Does an Interior Door Cost? Typical costs: Prices start at about $20-$100 for a no-frills, interior door slab (no frame or hardware) or $40-$...
  [4] Score: 23.16
      for commercial concrete floors the cost is anywhere from $ 50 $ 1 00 dollar per sq ft more if you re trying to figure what the cost for concrete floor...
  [5] Score: 23.06
      The cost of the flooring, which ranges from about $4.50 per square foot and up. The cost of labor which typica